# Exercício: Spark Streaming — Bolsa de Valores → MySQL

**Objetivo:** coletar cotações de ações da B3, processar em micro-batches com **Spark Structured Streaming** e persistir no **MySQL local instalado no Colab**.

| Etapa | Tecnologia |
|-------|------------|
| Coleta | `yfinance` |
| Stream simulado | arquivos JSON monitorados pelo Spark |
| Processamento | PySpark Structured Streaming |
| Destino | MySQL local (`127.0.0.1`) via JDBC |

> O MySQL é instalado e configurado automaticamente neste notebook — **não precisa de Azure nem secrets externos**.

## 1. Instalação de dependências

In [ ]:
!pip -q install pyspark yfinance pymysql

# Instala MySQL Server diretamente na VM do Colab
!apt-get -qq update
!DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mysql-server > /dev/null
!service mysql start

import time
time.sleep(3)
print("✅ Dependências instaladas e MySQL iniciado.")

## 2. Configurar MySQL local no Colab

Cria banco, usuário e credenciais usadas pelo Spark e pelo `pymysql`.

In [ ]:
import os
import json
import time
import threading
import subprocess
from datetime import datetime
from pathlib import Path

import yfinance as yf
import pymysql

# --- MySQL local no Colab (sem Azure) ---
MYSQL_HOST = "127.0.0.1"
MYSQL_PORT = 3306
MYSQL_DATABASE = "bolsa"
MYSQL_USER = "spark"
MYSQL_PASSWORD = "spark123"
MYSQL_ROOT_PASSWORD = "colab_root"

SETUP_SQL = f"""
ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY '{MYSQL_ROOT_PASSWORD}';
CREATE DATABASE IF NOT EXISTS `{MYSQL_DATABASE}` CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;
CREATE USER IF NOT EXISTS '{MYSQL_USER}'@'localhost' IDENTIFIED WITH mysql_native_password BY '{MYSQL_PASSWORD}';
CREATE USER IF NOT EXISTS '{MYSQL_USER}'@'127.0.0.1' IDENTIFIED WITH mysql_native_password BY '{MYSQL_PASSWORD}';
GRANT ALL PRIVILEGES ON `{MYSQL_DATABASE}`.* TO '{MYSQL_USER}'@'localhost';
GRANT ALL PRIVILEGES ON `{MYSQL_DATABASE}`.* TO '{MYSQL_USER}'@'127.0.0.1';
FLUSH PRIVILEGES;
"""

proc = subprocess.run(
    ["mysql", "-u", "root"],
    input=SETUP_SQL,
    text=True,
    capture_output=True,
)
if proc.returncode != 0:
    raise RuntimeError(f"Erro ao configurar MySQL:\n{proc.stderr}")

# --- Ações B3 ---
TICKERS = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "ABEV3.SA"]

# --- Simulação de stream ---
STREAM_DIR = Path("/content/stock_stream")
STREAM_DIR.mkdir(parents=True, exist_ok=True)
INTERVALO_SEGUNDOS = 10
DURACAO_PRODUTOR_SEG = 120

JDBC_URL = (
    f"jdbc:mysql://{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
    f"?useSSL=false&allowPublicKeyRetrieval=true&serverTimezone=UTC"
)
MYSQL_CONNECTOR_JAR = "/content/mysql-connector-j-8.3.0.jar"

print(f"MySQL local: {MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}")
print(f"Usuário: {MYSQL_USER}")
print(f"Stream dir: {STREAM_DIR}")

## 3. Baixar driver JDBC e criar tabela `cotacoes`

In [ ]:
!wget -q -O {MYSQL_CONNECTOR_JAR} https://repo1.maven.org/maven2/com/mysql/mysql-connector-j/8.3.0/mysql-connector-j-8.3.0.jar

CREATE_TABLE_SQL = """
CREATE TABLE IF NOT EXISTS cotacoes (
    id            BIGINT AUTO_INCREMENT PRIMARY KEY,
    ticker        VARCHAR(20)  NOT NULL,
    preco         DECIMAL(18,4) NOT NULL,
    abertura      DECIMAL(18,4),
    maxima        DECIMAL(18,4),
    minima        DECIMAL(18,4),
    volume        BIGINT,
    moeda         VARCHAR(10),
    bolsa         VARCHAR(50),
    coletado_em   DATETIME     NOT NULL,
    processado_em TIMESTAMP    DEFAULT CURRENT_TIMESTAMP,
    INDEX idx_ticker_data (ticker, coletado_em)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
"""

conn = pymysql.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    database=MYSQL_DATABASE,
    charset="utf8mb4",
)
try:
    with conn.cursor() as cur:
        cur.execute(CREATE_TABLE_SQL)
    conn.commit()
    print("✅ Tabela `cotacoes` pronta no MySQL local.")
finally:
    conn.close()

## 4. Produtor — simula stream de cotações

Em produção a fonte seria Kafka ou WebSocket. Aqui gravamos JSONs em disco para o Spark consumir.

In [ ]:
def coletar_cotacao(ticker: str) -> dict | None:
    """Busca cotação atual via yfinance."""
    try:
        info = yf.Ticker(ticker)
        fast = info.fast_info
        preco = fast.last_price if hasattr(fast, "last_price") else None
        if preco is None:
            hist = info.history(period="1d", interval="1m")
            if hist.empty:
                return None
            preco = float(hist["Close"].iloc[-1])
            abertura = float(hist["Open"].iloc[0])
            maxima = float(hist["High"].max())
            minima = float(hist["Low"].min())
            volume = int(hist["Volume"].sum())
        else:
            abertura = getattr(fast, "open", None)
            maxima = getattr(fast, "day_high", None)
            minima = getattr(fast, "day_low", None)
            volume = getattr(fast, "last_volume", None)

        return {
            "ticker": ticker,
            "preco": float(preco),
            "abertura": float(abertura) if abertura else None,
            "maxima": float(maxima) if maxima else None,
            "minima": float(minima) if minima else None,
            "volume": int(volume) if volume else None,
            "moeda": getattr(fast, "currency", "BRL") if hasattr(fast, "currency") else "BRL",
            "bolsa": "B3",
            "coletado_em": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
        }
    except Exception as exc:
        print(f"Erro ao coletar {ticker}: {exc}")
        return None


def produzir_stream(stop_event: threading.Event):
    """Grava um arquivo JSON por ticker a cada intervalo."""
    inicio = time.time()
    lote = 0
    while not stop_event.is_set() and (time.time() - inicio) < DURACAO_PRODUTOR_SEG:
        lote += 1
        ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
        for ticker in TICKERS:
            registro = coletar_cotacao(ticker)
            if registro:
                arquivo = STREAM_DIR / f"lote{lote}_{ticker.replace('.', '_')}_{ts}.json"
                arquivo.write_text(json.dumps(registro), encoding="utf-8")
                print(f"📥 {arquivo.name} → R$ {registro['preco']:.2f}")
        time.sleep(INTERVALO_SEGUNDOS)
    print("🏁 Produtor encerrado.")


# Teste rápido de uma cotação
amostra = coletar_cotacao(TICKERS[0])
print("Amostra:", amostra)

## 5. Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, LongType,
)

spark = (
    SparkSession.builder
    .appName("BolsaStreamingMySQL")
    .config("spark.jars", MYSQL_CONNECTOR_JAR)
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}")

## 6. Structured Streaming — ler JSON e gravar no MySQL

**Exercício:** observe o schema inferido e complete a transformação se quiser adicionar colunas calculadas (ex.: variação % em relação à abertura).

In [ ]:
schema = StructType([
    StructField("ticker", StringType(), False),
    StructField("preco", DoubleType(), False),
    StructField("abertura", DoubleType(), True),
    StructField("maxima", DoubleType(), True),
    StructField("minima", DoubleType(), True),
    StructField("volume", LongType(), True),
    StructField("moeda", StringType(), True),
    StructField("bolsa", StringType(), True),
    StructField("coletado_em", StringType(), False),
])


def escrever_mysql(batch_df, batch_id: int):
    """Callback foreachBatch: grava micro-batch no MySQL."""
    if batch_df.isEmpty():
        return

    df_mysql = (
        batch_df
        .withColumn("coletado_em", to_timestamp(col("coletado_em")))
        .select(
            "ticker", "preco", "abertura", "maxima", "minima",
            "volume", "moeda", "bolsa", "coletado_em",
        )
    )

    count = df_mysql.count()
    print(f"Batch {batch_id}: gravando {count} registro(s) no MySQL...")

    (
        df_mysql.write
        .format("jdbc")
        .option("url", JDBC_URL)
        .option("dbtable", "cotacoes")
        .option("user", MYSQL_USER)
        .option("password", MYSQL_PASSWORD)
        .option("driver", "com.mysql.cj.jdbc.Driver")
        .mode("append")
        .save()
    )


stream_df = (
    spark.readStream
    .schema(schema)
    .option("maxFilesPerTrigger", 10)
    .json(str(STREAM_DIR))
)

query = (
    stream_df.writeStream
    .foreachBatch(escrever_mysql)
    .option("checkpointLocation", "/content/checkpoint_bolsa")
    .trigger(processingTime="15 seconds")
    .start()
)

print(f"Streaming query id: {query.id}")

## 7. Executar produtor + aguardar streaming

In [ ]:
stop_event = threading.Event()
produtor = threading.Thread(target=produzir_stream, args=(stop_event,), daemon=True)
produtor.start()

# Aguarda produtor + margem para o Spark processar o último lote
time.sleep(DURACAO_PRODUTOR_SEG + 30)

stop_event.set()
produtor.join(timeout=5)

query.stop()
print("✅ Pipeline concluído.")

## 8. Validar dados no MySQL

In [ ]:
conn = pymysql.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    database=MYSQL_DATABASE,
    charset="utf8mb4",
)

with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM cotacoes")
    total = cur.fetchone()[0]
    print(f"Total de registros: {total}")

    cur.execute("""
        SELECT ticker, preco, coletado_em
        FROM cotacoes
        ORDER BY coletado_em DESC
        LIMIT 10
    """)
    print("\nÚltimas 10 cotações:")
    for row in cur.fetchall():
        print(f"  {row[0]:12} R$ {row[1]:>10.2f}  {row[2]}")

conn.close()

## 9. Desafios extras (opcional)

1. **Agregação streaming:** calcule média móvel de preço por ticker usando `groupBy` + window de 1 minuto.
2. **Alerta:** dispare log quando `preco` cair mais de 2% em relação à `abertura`.
3. **Kafka real:** substitua a pasta JSON por um tópico Kafka e use `readStream.format("kafka")`.
4. **Dashboard:** conecte Grafana ou Metabase ao MySQL para visualizar as cotações em tempo real.

In [ ]:
# Encerrar Spark ao finalizar
spark.stop()